In [2]:
import torch
import pandas as pd
def cov(thetas, sigmas, N, ATT):

    ub = thetas + 1.96 * sigmas/ torch.sqrt(torch.tensor(N))
    lb = thetas - 1.96 * sigmas / torch.sqrt(torch.tensor(N))
    coverage = torch.mean( ( (ATT >= lb) & (ATT <= ub) ).float() , 0 )
    interval_length = torch.mean(ub - lb,0)
    return {"Coverage":coverage,
            "interval_length": interval_length}

In [3]:
N = 500
# List of propensity models
#models = ['logistic', 'truncated_logistic', 'truncated_adv']
models = ['logistic']
# Create lists to store results
model_names = []
rmse_linear_list = []
sd_linear_list = []
rmse_riesz_list = []
sd_riesz_list = []
mea_linear_list = []
mea_riesz_list = []
coverage_linear_list = []
coverage_riesz_list = []
int_length_linear_list = []
int_length_riesz_list = []

# Read and compute RMSE, SD, Coverage, and Interval Length for each model
for model in models:
    file_suffix = f"N:{N}_{model}"

    pred_theta = torch.load(f"results/{file_suffix}_pred_theta.pt")
    pred_sig = torch.load(f"results/{file_suffix}_pred_sig.pt")
    ATT = torch.load(f"results/{file_suffix}_ATT_calculations.pt")["ATT"]

    rows = []

    # Define method indices and names: (index in pred_theta, method name)
    method_info = [
        (1, "Linear"),
        (2, "LASSO Riesz"),
        (3, "RF Riesz"),
        (4, "Net Riesz")
    ]

    for idx, method_name in method_info:
        est = pred_theta[:, idx]
        sig = pred_sig[:, idx]

        bias = (est - ATT).mean().item()
        rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
        cov_result = cov(est, sig, N, ATT)
        coverage = cov_result["Coverage"].item()
        int_len = cov_result["interval_length"].item()

        rows.append([method_name, bias, rmse, coverage, int_len])

    # Display results in a DataFrame
    df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
    
    print(model)
    print(df.to_string(index=False))


logistic
     Method      Bias     RMSE  Coverage  Interval Length
     Linear -0.100996 0.320759      0.62         0.643171
LASSO Riesz  0.056507 0.265902      0.98         1.194528
   RF Riesz  0.030652 0.314378      0.94         1.184297
  Net Riesz -0.017761 0.353046      0.95         1.337447
